# The baseline closed loop, step by step

One real shot through the simplest loop: syndrome data in -> controller (pulses to binary) -> syndrome packing -> Buffer 0 -> window manager -> weak decoder (its own memory; fetch, algorithm, release at a clock) -> Pauli frame. No reorder buffer, no strong tier, no confidence estimator.

Every number shown is measured inside the simulator on this run. The configured costs all come from one file, `experiments/baseline/baseline_closed_loop.yaml`.


In [1]:
import os
import sys

notebook_directory = os.getcwd()
repo_from_notebook_directory = os.path.join(notebook_directory, "..", "..")
running_inside_baseline_folder = notebook_directory.endswith("baseline")
if running_inside_baseline_folder:
    REPO = os.path.abspath(repo_from_notebook_directory)
else:
    REPO = notebook_directory
sys.path.insert(0, REPO)
os.chdir(REPO)

from decsim.config import microseconds


def row_as_strings(row: dict, columns: list) -> list:
    """One table row: the row's value under each column, as text."""
    values = []
    for column in columns:
        value = row.get(column, "")
        values.append(str(value))
    return values


def column_widths(columns: list, cells: list) -> list:
    """Width of each column: its header or its widest cell."""
    widths = []
    for index, column in enumerate(columns):
        width = len(column)
        for row in cells:
            width = max(width, len(row[index]))
        widths.append(width)
    return widths


def aligned_line(values: list, widths: list) -> str:
    """One printed line: each value padded to its column width."""
    padded = []
    for value, width in zip(values, widths):
        padded.append(value.ljust(width))
    return "  ".join(padded)


def table(rows: list, columns: list) -> None:
    """Print rows (dicts) as an aligned text table with the given columns."""
    cells = []
    for row in rows:
        cells.append(row_as_strings(row, columns))
    widths = column_widths(columns, cells)
    rules = []
    for width in widths:
        rules.append("-" * width)
    print(aligned_line(columns, widths))
    print(aligned_line(rules, widths))
    for row in cells:
        print(aligned_line(row, widths))


print("repo:", REPO)


repo: /scratch/gpfs/MARTONOSI/sk2415/qlx-qec-sandbox/decsim


## 1. The configuration: every cost in one place

Round period, decoder card, engine cycles and clock, controller times, the link cards, the Pauli frame commit. Nothing else in the loop carries a number.


In [2]:
CONFIG_PATH = "experiments/baseline/baseline_closed_loop.yaml"
print(open(CONFIG_PATH).read())


# Baseline closed loop: the single source of every parameter of the experiment.
# Every number that costs simulated time carries its source next to it.

code_task: surface_code:rotated_memory_z     # stim generator task (real detector data)
distance: 3
rounds_per_shot: 60                          # QEC rounds per shot; windows are (commit d, buffer d) sliding
noise_probability: 0.001                     # all four stim noise channels
seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]        # one shot per seed at each sweep point

# Sweep axis 1: input round period (the QEC cycle time seen by the controller).
round_period_us: [1.0, 0.5, 0.2, 0.1, 0.05, 0.02]
# Sweep axis 2: weak decoder algorithm latency (one window), microseconds, or "measured":
# 0.028 = LILLIPUT [d=3, m=2] 7 cycles at 250 MHz, 2108.06569 sec. 6.3 and Table 4 (ASIC card).
# measured = wall clock of each real PyMatching call on this host (software decoder row).
algorithm_latency_us: [0.028, 0.28, measured]

controller:
  t_binary_